In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

In [2]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
model = AutoModel.from_pretrained("microsoft/deberta-v3-large")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

ValueError: Converting from Tiktoken failed, if a converter for SentencePiece is available, provide a model path with a SentencePiece tokenizer.model file.Currently available slow->fast convertors: ['AlbertTokenizer', 'BartTokenizer', 'BarthezTokenizer', 'BertTokenizer', 'BigBirdTokenizer', 'BlenderbotTokenizer', 'CamembertTokenizer', 'CLIPTokenizer', 'CodeGenTokenizer', 'ConvBertTokenizer', 'DebertaTokenizer', 'DebertaV2Tokenizer', 'DistilBertTokenizer', 'DPRReaderTokenizer', 'DPRQuestionEncoderTokenizer', 'DPRContextEncoderTokenizer', 'ElectraTokenizer', 'FNetTokenizer', 'FunnelTokenizer', 'GPT2Tokenizer', 'HerbertTokenizer', 'LayoutLMTokenizer', 'LayoutLMv2Tokenizer', 'LayoutLMv3Tokenizer', 'LayoutXLMTokenizer', 'LongformerTokenizer', 'LEDTokenizer', 'LxmertTokenizer', 'MarkupLMTokenizer', 'MBartTokenizer', 'MBart50Tokenizer', 'MPNetTokenizer', 'MobileBertTokenizer', 'MvpTokenizer', 'NllbTokenizer', 'OpenAIGPTTokenizer', 'PegasusTokenizer', 'Qwen2Tokenizer', 'RealmTokenizer', 'Refor

In [3]:
inputs = [
    ("You should probably", "TRAIN this model"),
    ("on a down-stream", "task to"),
    ("be able to use it for predictions", "and inference")
]

premises, hypotheses = zip(*inputs)

In [4]:
premises

('You should probably',
 'on a down-stream',
 'be able to use it for predictions')

In [5]:
tokenizer(premises[0], hypotheses[0])

NameError: name 'tokenizer' is not defined

In [35]:
tokenizer(premises[1], hypotheses[1])

{'input_ids': [0, 261, 10, 159, 12, 8656, 2, 2, 45025, 7, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [34]:
tokenizer(premises, hypotheses)

{'input_ids': [[0, 1185, 197, 1153, 2, 2, 565, 4396, 2444, 42, 1421, 2], [0, 261, 10, 159, 12, 8656, 2, 2, 45025, 7, 2], [0, 1610, 441, 7, 304, 24, 13, 12535, 2, 2, 463, 42752, 2]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [40]:
from typing import List, Tuple


class RoBERTa(nn.Module):
    def __init__(self, 
            model_name, 
            *args, 
            **kwargs
        ):
        super().__init__()

        self.tokenizer = RobertaTokenizer.from_pretrained(model_name)
        self.model = RobertaModel.from_pretrained(model_name)

    def forward(self, x: List[Tuple[str, str]]):

        premises, hypotheses = zip(*x)
        inputs = self.tokenizer(premises, hypotheses, return_tensors="pt", padding=True, truncation=True)

        inputs = {key: value.to(self.model.device) for key, value in inputs.items() if isinstance(value, torch.Tensor)}

        outputs = self.model(**inputs)

        sentence_embeddings = outputs.last_hidden_state[:, 0, :] # Shape: (batch_size, hidden_size)

        return sentence_embeddings


In [41]:
roberta = RoBERTa('roberta-base')

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [43]:
ret = roberta(inputs)

In [44]:
ret.shape

torch.Size([3, 768])